The code in this notebook was adapted from assignments I did for the Fundamentals of Natural Language Processing course by University of Colorado Boulder.

A function that turns a tokenized corpus: List[List[int]] into a list of bigrams: List[Tuple[int, int]]

In [ ]:
def get_bigrams(corpus: List[List[int]]) -> List[Tuple[int, int]]:
    bigram = ()
    bigrams = []
    for sentence in corpus:
        for word in range(len(sentence) - 1):
            bigram = (sentence[word], sentence[word + 1])
            bigrams.append(bigram)
    return bigrams

A function that calculates the conditional probability of one word occuring after another, i.e. the probability of a word occuring knowing the word that occured before it: P(word2 | word1) = Count(word1, word2)/Count(word1)

Without Laplace (add-one smoothing)

In [ ]:
def conditional_prob(w_2, w_1, corpus: List[List[int]]) -> float:
    """Return the conditional probability of `w_2` given `w_1` in `corpus`.
    P(w_2 | w_1) = count(w_1, w_2) / count(w_1)
    """
    
    bigrams = get_bigrams(corpus) #using the previous function
    """creating a list of tokens, i.e. unigrams"""
    tokens = [token for tokenized_sentence in corpus for token in tokenized_sentence]
    """counting the unique unigrams and bigrams"""
    unigram_counts = Counter(tokens)
    bigram_counts = Counter(bigrams)

    conditional_probability = bigram_counts[(w_1, w_2)]/unigram_counts[w_1]
    
    return conditional_probability

With Laplace (add-one smoothing):

To implement Laplace Smoothing, we need to modify our conditional probability function as follows:

P(word2|word1) = count(word1,word2)+1/count(word1)+V 

where V is the number of unique words in the corpus.

In [ ]:
def conditional_prob(w_2, w_1, corpus: List[List[int]]) -> float:
    
    bigrams = get_bigrams(corpus)
    tokens = [token for tokenized_sentence in corpus for token in tokenized_sentence]
    unigram_counts = Counter(tokens)
    bigram_counts = Counter(bigrams)

    """creating a list of unique tokens in the corpus, i.e. the vocabulary"""
    vocab = set(tokens)
    V = len(vocab)
    
    conditional_probability = (bigram_counts[(w_1, w_2)] + 1)/(unigram_counts[w_1] + V)
    
    
    return conditional_probability

Predicting the next word using conditional probability:

In [ ]:
def predict_next_word(w_1, corpus) -> int:

    """A dictionaty to keep tokens and their corresponding probabilities"""
    probs = {}
    for sentence in corpus:
        for token in sentence:
            cprob = conditional_prob(token, w_1, corpus)
            probs[token] = cprob
            
    most_likely_next_word = max(probs, key=probs.get)
        
    return most_likely_next_word

A function for predicting the next sentence. This function automatically stops if it reaches a given word limit, and it stops 
when it reaches '</s>' even if it hasn't reached the limit. 

In [ ]:
stop_token_id = token_to_id['</s>']
def predict_sentence_bigrams(initial_sequence: List[int], corpus: List[List[int]], limit: int = 5) -> List[int]:
    """Return a sentence of `len(initial_sequence)`+`limit` words 
    predicted from `initial_sequence` in `corpus`."""
    
    predicted_sequence = list(initial_sequence) #creating a copy of the initial sequence
    count = 0
    while count<=limit:
        initial_string = predicted_sequence[-1]
        predicted_word = predict_next_word(initial_string, corpus)
        if predicted_word == stop_token_id:
            break
        predicted_sequence.append(predicted_word)
        count+=1
                
    return predicted_sequence